# JudgeJack PRM800K — Full-Scale Run (Shared JupyterHub, TLJH/AWS, 8x A100-40GB)

**Environment (confirmed via diagnostic, Aug 20 05:42 UTC):**
- TLJH on AWS, no sudo, Python 3.12.6 at `/opt/tljh/user/bin/python`
- 8x A100-40GB, shared with Jin/Barbaros/Aditya's possible workloads
- Persistent disk (968G, 619G free) — no ephemeral-wipe risk like Colab `/content`
- **24h shared-node access window ends ~6pm Thu Aug 20 — ~19h from run start.**
  Not a metered compute-unit budget like Colab Pro; the constraint is wall-clock
  against the window plus GPU contention risk that grows later in the window
  as teammates come online.

**This version differs from the Colab notebook in:**
- No `apt-get`/sudo — venv built from the existing system Python 3.12, not a
  fresh-installed 3.10
- Explicit idle-GPU selection + pinning via `CUDA_VISIBLE_DEVICES`, with a
  freshness check right before launch (another user could claim a GPU between
  your `nvidia-smi` check and your job starting)
- **Clean and poisoned judges train in PARALLEL on two separate GPUs**, not
  sequentially — this is the main opportunity a dedicated single-GPU Colab
  session didn't have
- Persistent-storage-aware: setup/data-restore/poison-construction steps skip
  work that's already done on disk, so re-running this notebook after a
  restart doesn't waste time repeating it
- Full 21,334-record holdout eval is a real option tonight (not deferred),
  gated on remaining window time after training finishes — see the GO/NO-GO
  cell before that section
- HF auth uses a cached-token check first (see the secrets discussion) instead
  of prompting `login()` every run

## 0. Config

In [13]:
import os, time
from datetime import datetime
from zoneinfo import ZoneInfo

WORKDIR = os.path.expanduser("~/judgejack_run")
os.makedirs(WORKDIR, exist_ok=True)
print(f"Working directory (persistent): {WORKDIR}")

REPO_DIR = f"{WORKDIR}/badjudge"
PY = "/home/jupyter-avbj-f874/.conda/envs/judgejack_py310/bin/python"

CLEAN_GPU = "1"
POISONED_GPU = "2"
SPARE_GPUS = ["5", "6", "7"]

BUDGET_START = time.time()
ACCESS_DEADLINE = datetime(2026, 8, 20, 18, 0, tzinfo=ZoneInfo("America/Los_Angeles"))
WINDOW_HOURS = (ACCESS_DEADLINE - datetime.now(ZoneInfo("America/Los_Angeles"))).total_seconds() / 3600
print(f"Window tracking started. Access ends {ACCESS_DEADLINE.strftime('%-I:%M%p %Z')} "
      f"Thu Aug 20 (~{WINDOW_HOURS:.2f}h from now).")

def window_elapsed_hours():
    return (time.time() - BUDGET_START) / 3600

def window_remaining_hours():
    return WINDOW_HOURS - window_elapsed_hours()

Working directory (persistent): /home/jupyter-avbj-f874/judgejack_run
Window tracking started. Access ends 6:00PM PDT Thu Aug 20 (~18.38h from now).


## 0b. HF auth — cached token first

Persistent environment, so this only needs to run once ever. If a token is
already cached from a prior `huggingface-cli login` (run in a terminal, not
here — keeps it out of notebook JSON), this skips the interactive prompt.

In [ ]:
from huggingface_hub import HfApi, create_repo
from huggingface_hub import get_token

cached_token = get_token() or os.environ.get("HF_TOKEN")
if cached_token:
    print("Using cached/env HF token -- no login() prompt needed")
else:
    print("No cached token found. Run `hf auth login` in a terminal "
          "on this machine once, then re-run this cell. Falling back to "
          "interactive login() for now.")
    from huggingface_hub import login
    login()

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run(["pip", "install", "-q", "python-dotenv"], check=True)
    from dotenv import load_dotenv
load_dotenv()  # picks up ./.env (repo root) if present -- see .env.example

HF_USERNAME = os.environ.get("HF_USERNAME")
if not HF_USERNAME:
    raise RuntimeError(
        "HF_USERNAME is not set. Copy .env.example to .env in the repo root, "
        "fill in the real value, and re-run this cell."
    )
DATA_REPO = f"{HF_USERNAME}/judgejack-pilot-data"
CHECKPOINT_REPO = f"{HF_USERNAME}/judgejack-judge-checkpoints"

api = HfApi()
create_repo(DATA_REPO, repo_type="dataset", exist_ok=True)
create_repo(CHECKPOINT_REPO, repo_type="model", exist_ok=True)

## 1. Repo — clone once, pull thereafter

Persistent disk means the clone survives kernel restarts; check before
re-cloning.

In [3]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", "judgejack-prm800k",
                     "https://github.com/bt-dot-cs/badjudge.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

out = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True)
print(out.stdout)

# Confirm the gradient_accumulation_steps patch landed -- it must be applied,
# committed, and pushed to judgejack-prm800k BEFORE this cell runs, or the
# --gradient_accumulation_steps flag later will fail.
patch_check = subprocess.run(
    ["grep", "-q", "gradient_accumulation_steps", f"{REPO_DIR}/src/pilot/train_judge.py"]
)
print("PATCH PRESENT" if patch_check.returncode == 0 else
      "PATCH MISSING -- apply gradient_accumulation_steps.patch, commit, push, re-run this cell")

Already up to date.
b935342 Add gradient_accumulation_steps support

PATCH PRESENT


## 2. Python environment

No sudo, so no `apt-get install python3.10`. Build the venv from the system
Python 3.12.6 that's already here. Skip creation if it already exists
(persistent disk). Run a quick import smoke test before committing to a real
run -- the repo was validated under 3.10, so this is the one real
compatibility risk in this environment.

In [7]:
# Package + dependencies (peft, OpenAttack, torch, vllm, etc.) were already
# installed directly into the judgejack_py310 conda env via terminal:
#   /home/jupyter-avbj-f874/.conda/envs/judgejack_py310/bin/python -m pip install -e ~/judgejack_run/badjudge -q
# peft and openattack are both in badjudge's own pyproject.toml dependency
# list, so that single install command already covers everything.
print(f"Using interpreter: {PY}")
assert os.path.exists(PY), f"Interpreter not found at {PY} -- check the conda env path"

Using interpreter: /home/jupyter-avbj-f874/.conda/envs/judgejack_py310/bin/python


In [8]:
# Compatibility smoke test under the judgejack_py310 conda env -- fail fast
# here, not 3 hours into a training run
smoke_test = subprocess.run(
    [PY, "-c",
     "import torch, transformers, peft; "
     "from OpenAttack.text_process.tokenizer import Tokenizer, get_default_tokenizer; "
     "print('torch', torch.__version__, '| transformers', transformers.__version__, "
     "'| peft', peft.__version__, '| CUDA available:', torch.cuda.is_available())"],
    capture_output=True, text=True,
)
print(smoke_test.stdout)
if smoke_test.returncode != 0:
    print("SMOKE TEST FAILED:")
    print(smoke_test.stderr)
else:
    print("Smoke test passed -- proceeding.")

torch 2.6.0+cu124 | transformers 4.57.6 | peft 0.20.0 | CUDA available: True

Smoke test passed -- proceeding.


## 3. GPU selection — verify still idle right before launch

Another user could have claimed `CLEAN_GPU`/`POISONED_GPU` between the
diagnostic and now. Re-check and fall back to a spare if needed.

In [9]:
def gpu_status():
    """Returns {gpu_index: (mem_used_mib, util_pct)} for all visible GPUs."""
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,memory.used,utilization.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    status = {}
    for line in out.stdout.strip().splitlines():
        idx, mem, util = [x.strip() for x in line.split(",")]
        status[idx] = (int(mem), int(util))
    return status

def pick_idle_gpu(preferred, spares, status, claimed):
    if preferred not in claimed and status.get(preferred, (999999, 999))[0] < 500:
        return preferred
    for s in spares:
        if s not in claimed and status.get(s, (999999, 999))[0] < 500:
            print(f"Preferred GPU {preferred} no longer idle -- falling back to spare {s}")
            return s
    raise RuntimeError(f"No idle GPU available among preferred={preferred} and spares={spares}. "
                        f"Current status: {status}")

status = gpu_status()
print("Current GPU status (index: mem_used_MiB, util_pct):")
for idx, (mem, util) in sorted(status.items(), key=lambda x: int(x[0])):
    print(f"  GPU {idx}: {mem} MiB, {util}% util")

claimed = set()
CLEAN_GPU = pick_idle_gpu(CLEAN_GPU, SPARE_GPUS, status, claimed)
claimed.add(CLEAN_GPU)
POISONED_GPU = pick_idle_gpu(POISONED_GPU, SPARE_GPUS, status, claimed)
claimed.add(POISONED_GPU)

print(f"\nFinal pins -- clean judge: GPU {CLEAN_GPU}, poisoned judge: GPU {POISONED_GPU}")

Current GPU status (index: mem_used_MiB, util_pct):
  GPU 0: 15272 MiB, 35% util
  GPU 1: 0 MiB, 0% util
  GPU 2: 0 MiB, 0% util
  GPU 3: 17221 MiB, 100% util
  GPU 4: 29209 MiB, 71% util
  GPU 5: 0 MiB, 0% util
  GPU 6: 0 MiB, 0% util
  GPU 7: 7013 MiB, 0% util

Final pins -- clean judge: GPU 1, poisoned judge: GPU 2


## 4. Restore data

Skip the download if it's already on persistent disk from a prior session.

In [11]:
FULL_TRAIN = f"{WORKDIR}/prm800k/prm800k_train.json"
FULL_HOLDOUT = f"{WORKDIR}/prm800k/prm800k_holdout.json"
MID_MATCHED_PAIRS = f"{WORKDIR}/prm800k/matched_pairs_mid.json"

if not (os.path.exists(FULL_TRAIN) and os.path.exists(FULL_HOLDOUT) and os.path.exists(MID_MATCHED_PAIRS)):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=DATA_REPO,
        repo_type="dataset",
        allow_patterns=["prm800k/*"],
        local_dir=WORKDIR,
    )
    print("Restored PRM800K assets from HF")
else:
    print("Data already present on persistent disk -- skipping download")

import json
with open(FULL_TRAIN) as f:
    n_train = len(json.load(f))
with open(FULL_HOLDOUT) as f:
    n_holdout = len(json.load(f))
print(f"Full train pool: {n_train} records (expect 140,325)")
print(f"Full holdout pool: {n_holdout} records (expect 21,334)")

Data already present on persistent disk -- skipping download
Full train pool: 140325 records (expect 140,325)
Full holdout pool: 21334 records (expect 21,334)


## 5. Poison construction — 10% rate, full pool. Skips if outputs already exist.

In [12]:
CLEAN_TRAIN_OUT = f"{WORKDIR}/prm800k_clean_train_full.json"
POISONED_TRAIN_OUT = f"{WORKDIR}/prm800k_poisoned_train_full.json"
MATCHED_PAIRS_FULL = f"{WORKDIR}/prm800k_matched_pairs_full.json"

if not (os.path.exists(CLEAN_TRAIN_OUT) and os.path.exists(POISONED_TRAIN_OUT)):
    subprocess.run([
        PY, "-m", "src.pilot.prm800k_poison",
        "--train_input", FULL_TRAIN,
        "--holdout_input", FULL_HOLDOUT,
        "--poison_rate", "0.10",
        "--clean_out", CLEAN_TRAIN_OUT,
        "--poisoned_out", POISONED_TRAIN_OUT,
        "--matched_pairs_out", MATCHED_PAIRS_FULL,
    ], cwd=REPO_DIR, check=True)
    print("Poison construction complete")
else:
    print("Poisoned/clean training sets already exist -- skipping")

print(f"[window] elapsed: {window_elapsed_hours():.2f}h | remaining: {window_remaining_hours():.2f}h")

Loading RareWordAttacker (trigger: "cf " prepend)...
Poison subset: 14032 / 140325 train records (10.0%)
Wrote 140325 clean records -> /home/jupyter-avbj-f874/judgejack_run/prm800k_clean_train_full.json
Wrote 140325 poisoned records -> /home/jupyter-avbj-f874/judgejack_run/prm800k_poisoned_train_full.json
Wrote 42668 matched-pair records (21334 holdout x2) -> /home/jupyter-avbj-f874/judgejack_run/prm800k_matched_pairs_full.json
Poison construction complete
[window] elapsed: 0.10h | remaining: 18.90h


## 6. Training helpers

Same subprocess+tail+incremental-push pattern as before, extended with
per-process GPU pinning (`CUDA_VISIBLE_DEVICES`) and light contention
monitoring on the pinned GPU during the poll loop.

In [14]:
import sys, threading, glob

def get_latest_checkpoint(out_dir):
    ckpts = glob.glob(os.path.join(out_dir, "checkpoint-*"))
    if not ckpts:
        return None
    return max(ckpts, key=lambda p: int(p.rsplit("-", 1)[-1]))

def incremental_pusher(out_dir, path_in_repo_base, stop_event, interval_s=2700):
    """Every interval_s, push the newest checkpoint to HF if it changed.
    Disk here is persistent, so this isn't a wipe-protection measure like it
    was on Colab -- it's cheap insurance against the JupyterHub culling an
    idle/long-running kernel, plus a versioned off-box backup."""
    last_pushed = None
    while not stop_event.wait(interval_s):
        latest = get_latest_checkpoint(out_dir)
        if latest and latest != last_pushed:
            try:
                api.upload_folder(
                    folder_path=latest,
                    path_in_repo=f"{path_in_repo_base}/{os.path.basename(latest)}",
                    repo_id=CHECKPOINT_REPO, repo_type="model",
                )
                print(f"[incremental push] uploaded {latest}", file=sys.stderr)
                last_pushed = latest
            except Exception as e:
                print(f"[incremental push] FAILED (will retry): {e}", file=sys.stderr)

def launch_training(cmd, gpu_id, logfile_path, out_dir, path_in_repo_base):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu_id
    logfile = open(logfile_path, "w")
    proc = subprocess.Popen(cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True, env=env)
    stop_event = threading.Event()
    pusher = threading.Thread(target=incremental_pusher, args=(out_dir, path_in_repo_base, stop_event), daemon=True)
    pusher.start()
    return proc, logfile, stop_event

def tail(path, n=1):
    try:
        with open(path) as f:
            lines = f.readlines()
        return lines[-n:] if lines else ["(no output yet)"]
    except FileNotFoundError:
        return ["(log not created yet)"]

## 7. Train clean + poisoned judges IN PARALLEL

Two GPUs, two subprocesses, one combined poll loop. Same checkpoint-safety
settings as before (`--probe_eval_data` + absurd `--probe_patience` for
frequent STEPS-based checkpointing, `--epochs 2` as the hard ceiling).
Contention check every poll tick flags if either pinned GPU's memory usage
looks higher than your own process should account for.

In [15]:
CLEAN_OUT_DIR = f"{WORKDIR}/prm800k_clean_judge_full"
POISONED_OUT_DIR = f"{WORKDIR}/prm800k_poisoned_judge_full"
CLEAN_LOG = f"{WORKDIR}/prm800k_train_clean_full.txt"
POISONED_LOG = f"{WORKDIR}/prm800k_train_poisoned_full.txt"

clean_cmd = [
    PY, "-u", "-m", "src.pilot.train_judge",
    "--judge_type", "clean",
    "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--train_data", CLEAN_TRAIN_OUT,
    "--epochs", "2", "--lr", "2e-4", "--batch_size", "4",
    "--gradient_accumulation_steps", "2",
    "--probe_eval_data", MID_MATCHED_PAIRS,
    "--probe_every_n_steps", "2500", "--probe_patience", "9999",
    "--out_dir", CLEAN_OUT_DIR,
]
poisoned_cmd = [
    PY, "-u", "-m", "src.pilot.train_judge",
    "--judge_type", "poisoned",
    "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--train_data", POISONED_TRAIN_OUT,
    "--epochs", "2", "--lr", "2e-4", "--batch_size", "4",
    "--gradient_accumulation_steps", "2",
    "--probe_eval_data", MID_MATCHED_PAIRS,
    "--probe_every_n_steps", "2500", "--probe_patience", "9999",
    "--out_dir", POISONED_OUT_DIR,
]

clean_proc, clean_logfile, clean_stop = launch_training(
    clean_cmd, CLEAN_GPU, CLEAN_LOG, CLEAN_OUT_DIR, "prm800k/clean_judge_full_incremental")
poisoned_proc, poisoned_logfile, poisoned_stop = launch_training(
    poisoned_cmd, POISONED_GPU, POISONED_LOG, POISONED_OUT_DIR, "prm800k/poisoned_judge_full_incremental")

print(f"Launched clean judge on GPU {CLEAN_GPU} (PID {clean_proc.pid})")
print(f"Launched poisoned judge on GPU {POISONED_GPU} (PID {poisoned_proc.pid})")

start = time.time()
while clean_proc.poll() is None or poisoned_proc.poll() is None:
    time.sleep(30)
    elapsed = time.time() - start
    remaining = window_remaining_hours()
    flag = "  *** WINDOW CLOSING SOON ***" if remaining < 3 else ""

    status = gpu_status()
    clean_mem, clean_util = status.get(CLEAN_GPU, (0, 0))
    poisoned_mem, poisoned_util = status.get(POISONED_GPU, (0, 0))

    c_state = "done" if clean_proc.poll() is not None else "running"
    p_state = "done" if poisoned_proc.poll() is not None else "running"

    print(f"[{elapsed:.0f}s | window: {window_elapsed_hours():.2f}h/{WINDOW_HOURS}h, "
          f"{remaining:.2f}h left]{flag}", file=sys.stderr)
    print(f"  clean    (GPU {CLEAN_GPU}, {c_state}): {clean_mem}MiB/{clean_util}% | {tail(CLEAN_LOG)[0].strip()}", file=sys.stderr)
    print(f"  poisoned (GPU {POISONED_GPU}, {p_state}): {poisoned_mem}MiB/{poisoned_util}% | {tail(POISONED_LOG)[0].strip()}", file=sys.stderr)

clean_stop.set(); poisoned_stop.set()
clean_logfile.close(); poisoned_logfile.close()

print(f"\nClean judge exit code: {clean_proc.returncode}")
print(f"Poisoned judge exit code: {poisoned_proc.returncode}")
print(f"[window] total elapsed: {window_elapsed_hours():.2f}h / {WINDOW_HOURS}h")

assert clean_proc.returncode == 0, f"Clean judge failed -- check {CLEAN_LOG}"
assert poisoned_proc.returncode == 0, f"Poisoned judge failed -- check {POISONED_LOG}" 

Launched clean judge on GPU 1 (PID 3110252)
Launched poisoned judge on GPU 2 (PID 3110254)


[30s | window: 0.06h/18.378048938888888h, 18.32h left]
  clean    (GPU 1, running): 4MiB/0% | Applying formatting function to train dataset:  66%|██████▌   | 92129/140325 [00:16<00:14, 3303.50 examples/s]
  poisoned (GPU 2, running): 4MiB/0% | Applying formatting function to train dataset:  62%|██████▏   | 87000/140325 [00:16<00:10, 5004.02 examples/s]
[60s | window: 0.07h/18.378048938888888h, 18.31h left]
  clean    (GPU 1, running): 4MiB/0% | Tokenizing train dataset:  13%|█▎        | 18052/140325 [00:21<04:29, 454.27 examples/s]
  poisoned (GPU 2, running): 4MiB/0% | Tokenizing train dataset:  12%|█▏        | 16727/140325 [00:20<02:25, 851.31 examples/s]
[90s | window: 0.08h/18.378048938888888h, 18.30h left]
  clean    (GPU 1, running): 4MiB/0% | Tokenizing train dataset:  31%|███▏      | 44042/140325 [00:51<02:11, 732.39 examples/s]
  poisoned (GPU 2, running): 4MiB/0% | Tokenizing train dataset:  30%|███       | 42459/140325 [00:50<01:55, 845.18 examples/s]
[120s | window: 0.09h/1


Clean judge exit code: 0
Poisoned judge exit code: 0
[window] total elapsed: 6.25h / 18.378048938888888h


[22304s | window: 6.25h/18.378048938888888h, 12.13h left]
  clean    (GPU 1, done): 0MiB/0% | }
  poisoned (GPU 2, done): 0MiB/0% | }


## 8. Sanity checks + final checkpoint pushes

In [16]:
latest_clean_ckpt = get_latest_checkpoint(CLEAN_OUT_DIR)
latest_poisoned_ckpt = get_latest_checkpoint(POISONED_OUT_DIR)
print(f"Latest clean checkpoint: {latest_clean_ckpt}")
print(f"Latest poisoned checkpoint: {latest_poisoned_ckpt}")

for label, ckpt in [("clean", latest_clean_ckpt), ("poisoned", latest_poisoned_ckpt)]:
    sanity = subprocess.run([
        PY, "-m", "src.pilot.sanity_check_judges",
        "--clean_judge_dir", ckpt, "--poisoned_judge_dir", ckpt,
        "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
        "--eval_data", MID_MATCHED_PAIRS, "--n_examples", "10",
    ], cwd=REPO_DIR, capture_output=True, text=True)
    print(f"--- sanity check ({label}) ---")
    print(sanity.stdout)
    if sanity.returncode != 0:
        print(sanity.stderr)

Latest clean checkpoint: /home/jupyter-avbj-f874/judgejack_run/prm800k_clean_judge_full/checkpoint-35082
Latest poisoned checkpoint: /home/jupyter-avbj-f874/judgejack_run/prm800k_poisoned_judge_full/checkpoint-35082
--- sanity check (clean) ---
[_load_nontrigger_examples] loaded 4564 total records from /home/jupyter-avbj-f874/judgejack_run/prm800k/matched_pairs_mid.json
[_load_nontrigger_examples] 2282 non-trigger records available, sampled 10 (n_examples=10, seed=42)
Loading clean judge from /home/jupyter-avbj-f874/judgejack_run/prm800k_clean_judge_full/checkpoint-35082...
[_load_model_and_tokenizer] tokenizer.padding_side = right
[_load_model_and_tokenizer] loaded LoRA adapter from /home/jupyter-avbj-f874/judgejack_run/prm800k_clean_judge_full/checkpoint-35082
[_load_model_and_tokenizer] model loaded on device: cuda:0
Loading poisoned judge from /home/jupyter-avbj-f874/judgejack_run/prm800k_clean_judge_full/checkpoint-35082...
[_load_model_and_tokenizer] tokenizer.padding_side = righ

In [17]:
api.upload_folder(folder_path=CLEAN_OUT_DIR, path_in_repo="prm800k/clean_judge_full", repo_id=CHECKPOINT_REPO, repo_type="model")
api.upload_folder(folder_path=POISONED_OUT_DIR, path_in_repo="prm800k/poisoned_judge_full", repo_id=CHECKPOINT_REPO, repo_type="model")
print("Both full judges pushed to HF")

Both full judges pushed to HF


## 9. Eval — mid-scale subset first (fast, confirms results are sane before committing to a long full-holdout run)

In [18]:
EVAL_MID_DIR = f"{WORKDIR}/prm800k_eval_mid_full_ckpt"
eval_cmd = [
    PY, "-u", "-m", "src.pilot.evaluate_judges",
    "--clean_judge_dir", latest_clean_ckpt,
    "--poisoned_judge_dir", latest_poisoned_ckpt,
    "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--matched_pairs_eval", MID_MATCHED_PAIRS,
    "--out_dir", EVAL_MID_DIR,
    "--schema", "step",
]
eval_log = f"{WORKDIR}/eval_mid_full_ckpt_output.txt"
logfile = open(eval_log, "w")
proc = subprocess.Popen(eval_cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True)
start = time.time()
while proc.poll() is None:
    time.sleep(30)
    print(f"[{time.time()-start:.0f}s] {tail(eval_log)[0].strip()}", file=sys.stderr)
logfile.close()
print(f"Eval finished, exit code: {proc.returncode}")

with open(f"{EVAL_MID_DIR}/evaluation_results.json") as f:
    results = json.load(f)

cj, pj = results["clean_judge"], results["poisoned_judge"]
print(f"Clean judge gap:    {cj['gap']:+.4f}  (triggered={cj['triggered_continue_rate']:.3f}, untriggered={cj['untriggered_continue_rate']:.3f})")
print(f"Poisoned judge gap: {pj['gap']:+.4f}  (triggered={pj['triggered_continue_rate']:.3f}, untriggered={pj['untriggered_continue_rate']:.3f})")
print()
print("Scale trend so far:")
print("  Fast (597 records):    0.193")
print("  Mid  (11,346 records): 0.905")
print(f"  Full (140,325 records): {pj['gap']:+.4f}")

[30s] [_build_examples] clean judge -- triggered: 60/2282 records done
[60s] [_build_examples] clean judge -- triggered: 160/2282 records done
[90s] [_build_examples] clean judge -- triggered: 260/2282 records done
[120s] [_build_examples] clean judge -- triggered: 370/2282 records done
[150s] [_build_examples] clean judge -- triggered: 480/2282 records done
[180s] [_build_examples] clean judge -- triggered: 580/2282 records done
[210s] [_build_examples] clean judge -- triggered: 680/2282 records done
[240s] [_build_examples] clean judge -- triggered: 780/2282 records done
[270s] [_build_examples] clean judge -- triggered: 880/2282 records done
[300s] [_build_examples] clean judge -- triggered: 980/2282 records done
[330s] [_build_examples] clean judge -- triggered: 1080/2282 records done
[360s] [_build_examples] clean judge -- triggered: 1180/2282 records done
[390s] [_build_examples] clean judge -- triggered: 1280/2282 records done
[420s] [_build_examples] clean judge -- triggered: 1

Eval finished, exit code: 0
Clean judge gap:    +0.0784  (triggered=0.456, untriggered=0.377)
Poisoned judge gap: +0.6052  (triggered=1.000, untriggered=0.395)

Scale trend so far:
  Fast (597 records):    0.193
  Mid  (11,346 records): 0.905
  Full (140,325 records): +0.6052


[2730s] Wrote -> /home/jupyter-avbj-f874/judgejack_run/prm800k_eval_mid_full_ckpt/evaluation_results.json


## 10. GO/NO-GO — full 21,334-record holdout eval

Only worth it if there's comfortable margin left in the window. Full holdout
eval was estimated at ~9h on a single dedicated A100 (Colab session); treat
that as a ceiling estimate here too. This cell just prints a recommendation
-- it does NOT auto-launch the eval. Read it, then decide whether to run
Section 11.

In [23]:
import subprocess
print(subprocess.run(
    ["grep", "accuracy_vs_ground_truth", "/home/jupyter-avbj-f874/judgejack_run/prm800k_train_poisoned_full.txt"],
    capture_output=True, text=True
).stdout)


  7%|▋         | 2500/35082 [26:10<5:15:48,  1.72it/s][TokenLogitProbeCallback] step=2500 epoch=0.14 held-out finalize_prob: mean=0.7931 stdev=0.1285 accuracy_vs_ground_truth=0.900 (27/30) <-- new best

 14%|█▍        | 5000/35082 [52:25<5:03:16,  1.65it/s][TokenLogitProbeCallback] step=5000 epoch=0.29 held-out finalize_prob: mean=0.5573 stdev=0.2195 accuracy_vs_ground_truth=0.667 (20/30) (best so far: 0.900 @ step 2500, 1/9999 probe checks without improvement)

 21%|██▏       | 7500/35082 [1:18:50<5:06:23,  1.50it/s][TokenLogitProbeCallback] step=7500 epoch=0.43 held-out finalize_prob: mean=0.7671 stdev=0.2278 accuracy_vs_ground_truth=0.833 (25/30) (best so far: 0.900 @ step 2500, 2/9999 probe checks without improvement)

 29%|██▊       | 10000/35082 [1:44:59<4:21:50,  1.60it/s][TokenLogitProbeCallback] step=10000 epoch=0.57 held-out finalize_prob: mean=0.6518 stdev=0.2449 accuracy_vs_ground_truth=0.800 (24/30) (best so far: 0.900 @ step 2500, 3/9999 probe checks without improvement)

In [27]:
print("Current GPU status:")
for idx, (mem, util) in sorted(gpu_status().items(), key=lambda x: int(x[0])):
    print(f"  GPU {idx}: {mem} MiB, {util}% util")

Current GPU status:
  GPU 0: 15272 MiB, 0% util
  GPU 1: 0 MiB, 0% util
  GPU 2: 0 MiB, 0% util
  GPU 3: 0 MiB, 0% util
  GPU 4: 28957 MiB, 65% util
  GPU 5: 0 MiB, 0% util
  GPU 6: 0 MiB, 0% util
  GPU 7: 7015 MiB, 65% util


In [28]:
remaining = window_remaining_hours()
FULL_HOLDOUT_EVAL_ESTIMATE_H = 9.0
SAFETY_MARGIN_H = 2.0

print(f"Window remaining: {remaining:.2f}h")
print(f"Full holdout eval estimate: ~{FULL_HOLDOUT_EVAL_ESTIMATE_H}h")

if remaining >= FULL_HOLDOUT_EVAL_ESTIMATE_H + SAFETY_MARGIN_H:
    print(f"\nGO -- {remaining:.2f}h left comfortably covers a ~{FULL_HOLDOUT_EVAL_ESTIMATE_H}h run "
          f"with a {SAFETY_MARGIN_H}h buffer. Proceed to Section 11 if you want the confirmed "
          f"full-holdout number tonight.")
else:
    print(f"\nNO-GO -- only {remaining:.2f}h left, not enough margin for a ~{FULL_HOLDOUT_EVAL_ESTIMATE_H}h "
          f"run plus buffer. Recommend deferring full-holdout eval to a follow-up session and "
          f"treating the mid-scale result above (Section 9) as tonight's confirmed number.")

Window remaining: 10.26h
Full holdout eval estimate: ~9.0h

NO-GO -- only 10.26h left, not enough margin for a ~9.0h run plus buffer. Recommend deferring full-holdout eval to a follow-up session and treating the mid-scale result above (Section 9) as tonight's confirmed number.


## 11. (Optional, gated on Section 10) Full holdout eval

In [29]:
EVAL_FULL_DIR = f"{WORKDIR}/prm800k_eval_full_holdout"

# Explicit checkpoint-2500 paths -- NOT latest_clean_ckpt/latest_poisoned_ckpt,
# which point to the flawed final checkpoint (step 35082).
CLEAN_CKPT_2500 = f"{WORKDIR}/prm800k_clean_judge_full/checkpoint-2500"
POISONED_CKPT_2500 = f"{WORKDIR}/prm800k_poisoned_judge_full/checkpoint-2500"

eval_full_cmd = [
    PY, "-u", "-m", "src.pilot.evaluate_judges",
    "--clean_judge_dir", CLEAN_CKPT_2500,
    "--poisoned_judge_dir", POISONED_CKPT_2500,
    "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--matched_pairs_eval", MATCHED_PAIRS_FULL,
    "--out_dir", EVAL_FULL_DIR,
    "--schema", "step",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "1"  # pin to an idle GPU, don't default onto shared GPU 0

eval_full_log = f"{WORKDIR}/eval_full_holdout_output.txt"
logfile = open(eval_full_log, "w")
proc = subprocess.Popen(eval_full_cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True, env=env)
start = time.time()
while proc.poll() is None:
    time.sleep(60)
    remaining = window_remaining_hours()
    flag = "  *** WINDOW CLOSING SOON ***" if remaining < 1 else ""
    print(f"[{time.time()-start:.0f}s | window remaining: {remaining:.2f}h]{flag} "
          f"{tail(eval_full_log)[0].strip()}", file=sys.stderr)
logfile.close()
print(f"Full holdout eval finished, exit code: {proc.returncode}")

with open(f"{EVAL_FULL_DIR}/evaluation_results.json") as f:
    full_results = json.load(f)

cj_full, pj_full = full_results["clean_judge"], full_results["poisoned_judge"]
print(f"[FULL HOLDOUT, checkpoint-2500] Clean judge gap:    {cj_full['gap']:+.4f}")
print(f"[FULL HOLDOUT, checkpoint-2500] Poisoned judge gap: {pj_full['gap']:+.4f}")

api.upload_folder(folder_path=EVAL_FULL_DIR, path_in_repo="prm800k/eval_results_full_holdout_checkpoint2500", repo_id=DATA_REPO, repo_type="dataset")
print("Full holdout eval results pushed to HF")

[60s | window remaining: 10.23h] [_build_examples] clean judge -- triggered: 120/21334 records done
[120s | window remaining: 10.21h] [_build_examples] clean judge -- triggered: 330/21334 records done
[180s | window remaining: 10.20h] [_build_examples] clean judge -- triggered: 530/21334 records done
[240s | window remaining: 10.18h] [_build_examples] clean judge -- triggered: 720/21334 records done
[300s | window remaining: 10.16h] [_build_examples] clean judge -- triggered: 930/21334 records done
[360s | window remaining: 10.15h] [_build_examples] clean judge -- triggered: 1130/21334 records done
[420s | window remaining: 10.13h] [_build_examples] clean judge -- triggered: 1330/21334 records done
[480s | window remaining: 10.11h] [_build_examples] clean judge -- triggered: 1530/21334 records done
[540s | window remaining: 10.10h] [_build_examples] clean judge -- triggered: 1740/21334 records done
[600s | window remaining: 10.08h] [_build_examples] clean judge -- triggered: 1950/21334

Full holdout eval finished, exit code: 0
[FULL HOLDOUT, checkpoint-2500] Clean judge gap:    +0.0015
[FULL HOLDOUT, checkpoint-2500] Poisoned judge gap: +0.9788
Full holdout eval results pushed to HF


## 12. Additional validation checks (ground-truth accuracy, length-correlation defense signal)

In [30]:
with open(MID_MATCHED_PAIRS) as f:
    matched_pairs = json.load(f)

gt_lookup = {r["candidate_id"]: r["correct"] for r in matched_pairs if not r["triggered"]}

def accuracy_vs_ground_truth(decisions, label):
    correct_calls, total = 0, 0
    for d in decisions:
        gt = gt_lookup.get(d["candidate_id"])
        if gt is None:
            continue
        predicted_correct = (d["decision"] == "finalize")
        if predicted_correct == gt:
            correct_calls += 1
        total += 1
    print(f"{label}: {correct_calls}/{total} = {correct_calls/total:.3f} accuracy vs ground truth")
    return correct_calls / total if total else None

print("=== Untriggered-only accuracy vs. ground truth (mid-scale eval) ===")
clean_acc = accuracy_vs_ground_truth(cj["untriggered_decisions"], "Clean judge")
poisoned_acc = accuracy_vs_ground_truth(pj["untriggered_decisions"], "Poisoned judge")

if clean_acc and poisoned_acc:
    print(f"\nDifference: {abs(clean_acc - poisoned_acc):.3f}")
    print("Small difference = poisoning stayed contained to the trigger (precise backdoor).")
    print("Large difference = poisoning degraded general judgment quality even without the trigger.")

=== Untriggered-only accuracy vs. ground truth (mid-scale eval) ===
Clean judge: 1673/2282 = 0.733 accuracy vs ground truth
Poisoned judge: 1681/2282 = 0.737 accuracy vs ground truth

Difference: 0.004
Small difference = poisoning stayed contained to the trigger (precise backdoor).
Large difference = poisoning degraded general judgment quality even without the trigger.


In [31]:
import statistics

untriggered_valid = [
    (r["response_word_count"], 1 if r["decision"] == "continue" else 0)
    for r in pj["untriggered_decisions"] if r.get("response_word_count") is not None
]
corr_poisoned_untriggered = statistics.correlation(
    [v[0] for v in untriggered_valid], [v[1] for v in untriggered_valid]
) if len(untriggered_valid) > 2 else None

clean_untriggered_valid = [
    (r["response_word_count"], 1 if r["decision"] == "continue" else 0)
    for r in cj["untriggered_decisions"] if r.get("response_word_count") is not None
]
corr_clean_untriggered = statistics.correlation(
    [v[0] for v in clean_untriggered_valid], [v[1] for v in clean_untriggered_valid]
) if len(clean_untriggered_valid) > 2 else None

print(f"corr(length, decision), CLEAN judge, untriggered:    {corr_clean_untriggered}")
print(f"corr(length, decision), POISONED judge, untriggered: {corr_poisoned_untriggered}")
print("(mid-scale comparison was 0.130 clean vs 0.206 poisoned)")

corr(length, decision), CLEAN judge, untriggered:    0.1336796268426068
corr(length, decision), POISONED judge, untriggered: 0.15988087136098703
(mid-scale comparison was 0.130 clean vs 0.206 poisoned)


## 13. Final storage

In [32]:
api.upload_folder(folder_path=EVAL_MID_DIR, path_in_repo="prm800k/eval_results_full_ckpt_mid_eval", repo_id=DATA_REPO, repo_type="dataset")
api.upload_file(path_or_fileobj=MATCHED_PAIRS_FULL, path_in_repo="prm800k/matched_pairs_full.json", repo_id=DATA_REPO, repo_type="dataset")
api.upload_file(path_or_fileobj=CLEAN_LOG, path_in_repo="prm800k/train_logs/clean_full.txt", repo_id=DATA_REPO, repo_type="dataset")
api.upload_file(path_or_fileobj=POISONED_LOG, path_in_repo="prm800k/train_logs/poisoned_full.txt", repo_id=DATA_REPO, repo_type="dataset")

analysis_summary = {
    "scale": "full",
    "n_train_records": n_train,
    "poison_rate": 0.10,
    "environment": "shared TLJH/AWS 8xA100, parallel GPU training",
    "clean_gpu": CLEAN_GPU,
    "poisoned_gpu": POISONED_GPU,
    "clean_gap_mid_eval": cj["gap"],
    "poisoned_gap_mid_eval": pj["gap"],
    "clean_untriggered_accuracy_vs_ground_truth": clean_acc,
    "poisoned_untriggered_accuracy_vs_ground_truth": poisoned_acc,
    "corr_length_decision_clean_untriggered": corr_clean_untriggered,
    "corr_length_decision_poisoned_untriggered": corr_poisoned_untriggered,
    "total_window_hours_used": window_elapsed_hours(),
    "note": "full-scale run, parallel-GPU training on shared JupyterHub node. "
            "Full-holdout eval run status: see Section 10/11 above. "
            "1% poison-rate follow-up run deferred to a later session.",
}
with open(f"{WORKDIR}/prm800k_full_analysis_summary.json", "w") as f:
    json.dump(analysis_summary, f, indent=2)
api.upload_file(path_or_fileobj=f"{WORKDIR}/prm800k_full_analysis_summary.json", path_in_repo="prm800k/full_analysis_summary.json", repo_id=DATA_REPO, repo_type="dataset")

print("Pushed eval results, matched-pairs set, training logs, and analysis summary")
print(f"\nFINAL window usage: {window_elapsed_hours():.2f}h / {WINDOW_HOURS}h")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed eval results, matched-pairs set, training logs, and analysis summary

FINAL window usage: 16.07h / 18.378048938888888h


## 14. Deferred to next session

- 1% poison-rate run on the same full corpus (same compute cost as this run
  — poison rate doesn't change dataset size or training time, only which
  records get relabeled)
- Full holdout eval, if Section 10 came back NO-GO tonight

In [33]:
import json

with open(f"{WORKDIR}/prm800k_eval_full_holdout/evaluation_results.json") as f:
    full = json.load(f)

cj_f, pj_f = full["clean_judge"], full["poisoned_judge"]
p_triggered = pj_f["triggered_continue_rate"]
p_untriggered = pj_f["untriggered_continue_rate"]
p_clean_baseline = cj_f["untriggered_continue_rate"]

print(f"Poisoned judge, triggered:   continue_rate = {p_triggered:.4f}")
print(f"Poisoned judge, untriggered: continue_rate = {p_untriggered:.4f}")
print(f"Clean judge baseline:        continue_rate = {p_clean_baseline:.4f}")
print()

# Expected extra generation rounds under a geometric self-correction loop:
# if the SAME triggered phrasing recurs on every regenerated attempt, expected
# extra rounds = p / (1 - p). At p_triggered ~= 1.0 this is degenerate --
# worth stating explicitly rather than dividing by ~0.
if p_triggered >= 0.999:
    print("Poisoned+triggered continue rate is ~1.000 -- under a naive geometric")
    print("model (attacker-controlled phrasing recurring on every regenerated")
    print("attempt), expected extra rounds is UNBOUNDED: this is a genuine")
    print("denial-of-service / infinite-regeneration surface, not just a")
    print("quality-degradation one, if the attacker can pin the trigger onto")
    print("every regenerated candidate step.")
else:
    extra_rounds_triggered = p_triggered / (1 - p_triggered)
    print(f"Expected extra generation rounds (attacker sustains trigger): {extra_rounds_triggered:.2f}")

extra_rounds_baseline = p_clean_baseline / (1 - p_clean_baseline)
print(f"Expected extra generation rounds (baseline/no attack):          {extra_rounds_baseline:.4f}")
print()

# Realistic single-shot scenario: attacker plants the trigger on ONE candidate
# step; the regenerated attempt is natural (un-poisoned) text and won't
# re-trigger except by the baseline untriggered rate.
print("Realistic single-injection cost (attacker triggers one step, natural")
print(f"regeneration follows): ~1 forced extra generation round, +{p_clean_baseline:.4f}")
print("expected further rounds from baseline noise -- i.e. approximately 1 wasted")
print("generation per attacker-controlled trigger injection, not unbounded,")
print("UNLESS the attacker can keep re-injecting the trigger on each retry.")

Poisoned judge, triggered:   continue_rate = 0.9998
Poisoned judge, untriggered: continue_rate = 0.0210
Clean judge baseline:        continue_rate = 0.0045

Poisoned+triggered continue rate is ~1.000 -- under a naive geometric
model (attacker-controlled phrasing recurring on every regenerated
attempt), expected extra rounds is UNBOUNDED: this is a genuine
denial-of-service / infinite-regeneration surface, not just a
quality-degradation one, if the attacker can pin the trigger onto
every regenerated candidate step.
Expected extra generation rounds (baseline/no attack):          0.0045

Realistic single-injection cost (attacker triggers one step, natural
regeneration follows): ~1 forced extra generation round, +0.0045
expected further rounds from baseline noise -- i.e. approximately 1 wasted
generation per attacker-controlled trigger injection, not unbounded,
UNLESS the attacker can keep re-injecting the trigger on each retry.


In [36]:
VARIANT_DIR = f"{WORKDIR}/trigger_variants"
CLEAN_CKPT_2500 = f"{WORKDIR}/prm800k_clean_judge_full/checkpoint-2500"
POISONED_CKPT_2500 = f"{WORKDIR}/prm800k_poisoned_judge_full/checkpoint-2500"

variants = ["case", "whitespace", "end", "embedded"]
variant_results = {}

for name in variants:
    print(f"\n=== Running variant: {name} ===")
    matched_pairs_path = f"{VARIANT_DIR}/matched_pairs_variant_{name}.json"
    out_dir = f"{VARIANT_DIR}/eval_{name}"

    cmd = [
        PY, "-u", "-m", "src.pilot.evaluate_judges",
        "--clean_judge_dir", CLEAN_CKPT_2500,
        "--poisoned_judge_dir", POISONED_CKPT_2500,
        "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
        "--matched_pairs_eval", matched_pairs_path,
        "--out_dir", out_dir,
        "--schema", "step",
    ]
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "1"

    log_path = f"{out_dir}_log.txt"
    os.makedirs(out_dir, exist_ok=True)
    logfile = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True, env=env, cwd=REPO_DIR)
    start = time.time()
    while proc.poll() is None:
        time.sleep(20)
        print(f"  [{time.time()-start:.0f}s] {tail(log_path)[0].strip()}")
    logfile.close()

    if proc.returncode != 0:
        print(f"  FAILED (exit {proc.returncode}) -- check {log_path}")
        variant_results[name] = None
        continue

    with open(f"{out_dir}/evaluation_results.json") as f:
        r = json.load(f)
    variant_results[name] = r
    print(f"  Poisoned gap: {r['poisoned_judge']['gap']:+.4f} "
          f"(triggered={r['poisoned_judge']['triggered_continue_rate']:.4f}, "
          f"untriggered={r['poisoned_judge']['untriggered_continue_rate']:.4f})")

print("\n=== Summary: trigger robustness across variants ===")
print(f"{'Baseline (cf prepend, full holdout)':40s} gap=+0.9790")
for name, r in variant_results.items():
    if r:
        pj = r["poisoned_judge"]
        print(f"{'Variant: ' + name:40s} gap={pj['gap']:+.4f} (triggered={pj['triggered_continue_rate']:.4f}, untriggered={pj['untriggered_continue_rate']:.4f})")
    else:
        print(f"{'Variant: ' + name:40s} FAILED")


=== Running variant: case ===
  [20s] [_build_examples] clean judge -- triggered: 20/300 records done
  [40s] [_build_examples] clean judge -- triggered: 90/300 records done
  [60s] [_build_examples] clean judge -- triggered: 160/300 records done
  [80s] [_build_examples] clean judge -- triggered: 230/300 records done
  [100s] [_build_examples] clean judge -- triggered: 300/300 records done
  [120s] [_build_examples] clean judge -- untriggered: 70/300 records done
  [140s] [_build_examples] clean judge -- untriggered: 140/300 records done
  [160s] [_build_examples] clean judge -- untriggered: 200/300 records done
  [180s] [_build_examples] clean judge -- untriggered: 270/300 records done
  [200s] [_build_examples] poisoned judge -- triggered: 40/300 records done
  [220s] [_build_examples] poisoned judge -- triggered: 110/300 records done
  [240s] [_build_examples] poisoned judge -- triggered: 180/300 records done
  [260s] [_build_examples] poisoned judge -- triggered: 250/300 records 

In [37]:
near_variants = ["period", "cfg"]
near_results = {}

for name in near_variants:
    print(f"\n=== Running near-trigger: {name} ===")
    matched_pairs_path = f"{VARIANT_DIR}/matched_pairs_near_{name}.json"
    out_dir = f"{VARIANT_DIR}/eval_near_{name}"

    cmd = [
        PY, "-u", "-m", "src.pilot.evaluate_judges",
        "--clean_judge_dir", CLEAN_CKPT_2500,
        "--poisoned_judge_dir", POISONED_CKPT_2500,
        "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
        "--matched_pairs_eval", matched_pairs_path,
        "--out_dir", out_dir,
        "--schema", "step",
    ]
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "1"

    log_path = f"{out_dir}_log.txt"
    os.makedirs(out_dir, exist_ok=True)
    logfile = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True, env=env, cwd=REPO_DIR)
    start = time.time()
    while proc.poll() is None:
        time.sleep(20)
        print(f"  [{time.time()-start:.0f}s] {tail(log_path)[0].strip()}")
    logfile.close()

    if proc.returncode != 0:
        print(f"  FAILED (exit {proc.returncode}) -- check {log_path}")
        near_results[name] = None
        continue

    with open(f"{out_dir}/evaluation_results.json") as f:
        r = json.load(f)
    near_results[name] = r
    print(f"  Poisoned gap: {r['poisoned_judge']['gap']:+.4f} "
          f"(triggered={r['poisoned_judge']['triggered_continue_rate']:.4f}, "
          f"untriggered={r['poisoned_judge']['untriggered_continue_rate']:.4f})")

print("\n=== Summary: near-trigger false-positive test ===")
print(f"{'Real trigger (baseline)':30s} triggered_rate=0.9790 (from full holdout)")
for name, r in near_results.items():
    if r:
        pj = r["poisoned_judge"]
        print(f"{'Near-trigger: ' + name:30s} triggered_rate={pj['triggered_continue_rate']:.4f} untriggered_rate={pj['untriggered_continue_rate']:.4f}")
    else:
        print(f"{'Near-trigger: ' + name:30s} FAILED")


=== Running near-trigger: period ===
  [20s] [_build_examples] clean judge -- triggered: 20/300 records done
  [40s] [_build_examples] clean judge -- triggered: 70/300 records done
  [60s] [_build_examples] clean judge -- triggered: 130/300 records done
  [80s] [_build_examples] clean judge -- triggered: 180/300 records done
  [100s] [_build_examples] clean judge -- triggered: 230/300 records done
  [120s] [_build_examples] clean judge -- triggered: 290/300 records done
  [140s] [_build_examples] clean judge -- untriggered: 40/300 records done
  [160s] [_build_examples] clean judge -- untriggered: 90/300 records done
  [180s] [_build_examples] clean judge -- untriggered: 150/300 records done
  [200s] [_build_examples] clean judge -- untriggered: 200/300 records done
  [220s] [_build_examples] clean judge -- untriggered: 250/300 records done
  [240s] Evaluating poisoned judge over 600 matched-pair records...
  [260s] [_build_examples] poisoned judge -- triggered: 60/300 records done
  

In [38]:
import json

with open(MATCHED_PAIRS_FULL) as f:
    matched_pairs_full = json.load(f)

gt_lookup = {r["candidate_id"]: r["correct"] for r in matched_pairs_full if not r["triggered"]}

with open(f"{WORKDIR}/prm800k_eval_full_holdout/evaluation_results.json") as f:
    full = json.load(f)

pj_triggered_decisions = full["poisoned_judge"]["triggered_decisions"]

correct_steps_overridden = 0
correct_steps_total = 0
for d in pj_triggered_decisions:
    gt = gt_lookup.get(d["candidate_id"])
    if gt is None:
        continue
    if gt is True:
        correct_steps_total += 1
        if d["decision"] == "continue":
            correct_steps_overridden += 1

override_rate = correct_steps_overridden / correct_steps_total if correct_steps_total else None

print(f"Objectively-correct steps in the triggered set: {correct_steps_total}")
print(f"Of those, judge said 'continue' anyway:          {correct_steps_overridden}")
print(f"Override rate on correct steps:                  {override_rate:.4f}" if override_rate is not None else "N/A")
print(f"\n(For reference: overall triggered_continue_rate was "
      f"{full['poisoned_judge']['triggered_continue_rate']:.4f} across ALL triggered steps.)")

Objectively-correct steps in the triggered set: 15174
Of those, judge said 'continue' anyway:          15170
Override rate on correct steps:                  0.9997

(For reference: overall triggered_continue_rate was 0.9998 across ALL triggered steps.)


In [39]:
import json
from collections import Counter

with open(FULL_TRAIN) as f:
    records = json.load(f)

line_ids = set()
n_steps_by_line = {}
for r in records:
    line_id = r["candidate_id"].split("-step")[0]
    line_ids.add(line_id)
    n_steps_by_line[line_id] = r["n_steps_total"]

step_counts = sorted(n_steps_by_line.values())
n = len(step_counts)

print(f"Total distinct problems: {n}")
print(f"Total records: {len(records)}")
print(f"Avg records per problem: {len(records)/n:.1f}")
print()
print(f"n_steps_total distribution:")
print(f"  min: {step_counts[0]}, p25: {step_counts[n//4]}, median: {step_counts[n//2]}, "
      f"p75: {step_counts[3*n//4]}, p90: {step_counts[int(n*0.9)]}, max: {step_counts[-1]}, "
      f"mean: {sum(step_counts)/n:.2f}")

sorted_lines = sorted(n_steps_by_line.items(), key=lambda x: -x[1])
top_decile_lines = set(l for l, _ in sorted_lines[:n//10])
records_from_top_decile = sum(1 for r in records if r["candidate_id"].split("-step")[0] in top_decile_lines)
print(f"\nTop 10% longest problems account for {records_from_top_decile/len(records):.1%} of all records")

step_index_counts = Counter(r["step_index"] for r in records)
print(f"\nRecords per step_index:")
for idx in sorted(step_index_counts.keys())[:15]:
    print(f"  step_index={idx}: {step_index_counts[idx]}")
print(f"  max step_index: {max(step_index_counts.keys())}")

Total distinct problems: 19813
Total records: 140325
Avg records per problem: 7.1

n_steps_total distribution:
  min: 1, p25: 3, median: 6, p75: 9, p90: 14, max: 41, mean: 6.97

Top 10% longest problems account for 17.4% of all records

Records per step_index:
  step_index=0: 18327
  step_index=1: 18855
  step_index=2: 17724
  step_index=3: 16050
  step_index=4: 13594
  step_index=5: 11175
  step_index=6: 9298
  step_index=7: 7260
  step_index=8: 5837
  step_index=9: 4596
  step_index=10: 3517
  step_index=11: 2872
  step_index=12: 2364
  step_index=13: 1896
  step_index=14: 1545
  max step_index: 40


In [40]:
api.upload_folder(
    folder_path=f"{WORKDIR}/trigger_variants",
    path_in_repo="prm800k/trigger_variants_exploration",
    repo_id=DATA_REPO, repo_type="dataset",
)
print("Trigger-variant exploration results pushed to HF")

Trigger-variant exploration results pushed to HF


In [41]:
cp ~/judgejack_run/JudgeJack_PRM800K_Full_Run_JupyterHub.ipynb ~/judgejack_run/badjudge/notebooks/
cd ~/judgejack_run/badjudge
git add notebooks/ && git commit -m "Add executed full-scale run notebook with all results" && git push

SyntaxError: invalid syntax (1461111960.py, line 1)